In [3]:
import numpy as np
import pandas as pd

In [4]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [5]:

df = pd.read_csv('covid_toy.csv')

In [6]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [7]:

df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
61,81,Female,98.0,Strong,Mumbai
18,64,Female,98.0,Mild,Bangalore
52,47,Female,100.0,Strong,Bangalore
65,69,Female,102.0,Mild,Bangalore
40,49,Female,102.0,Mild,Delhi
...,...,...,...,...,...
22,71,Female,98.0,Strong,Kolkata
12,25,Female,99.0,Strong,Kolkata
75,5,Male,102.0,Mild,Kolkata
9,64,Female,101.0,Mild,Delhi


## 1. Aam Zindagi

In [10]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [11]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [20]:
from sklearn.preprocessing import OneHotEncoder

# Step 1: Initialize encoder (correct for scikit-learn ≥1.2)
ohe = OneHotEncoder(drop='first', sparse_output=False)

# Step 2: Fit on training data and transform
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# Step 3: Transform test data
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])


In [13]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [22]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)

print(X_train_transformed.shape)
print(X_test_transformed.shape)


(80, 7)
(20, 7)


## 2.Mentos Zindagi

In [15]:

from sklearn.compose import ColumnTransformer

In [23]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')

In [24]:

transformer.fit_transform(X_train).shape

(80, 7)

In [25]:

transformer.transform(X_test).shape

(20, 7)